# EPF-Guide — Week 3 V1 Data Pipeline (Pathway A: PDF documents)

**Project:** EPF-Guide: KWSP Policy & Withdrawal Assistant  
**Author:** Kirthiggan A/L Saravanan (A24AI0042)  
**Course:** SAIA 3113 AI Engineering — Topic 3 Lab

**Why Pathway A only:** the Topic 2 requirements (FR1–FR3, FR5, NFR4, NFR5, RAI1) all depend on *official KWSP policy PDFs*. No image, audio, tabular or time-series data is needed, so Pathways B–F are intentionally not run.

**Topic 2 requirements this notebook serves**

| Requirement | What the data pipeline must provide |
|---|---|
| FR2 / FR3 / RAI1 | Text chunks that keep source file **and page number**, so every later answer can cite a verifiable source |
| FR5 / NFR5 | Per-document `doc_status`, version/date and access date, so only current documents are prepared |
| NFR4 | A repeatable pipeline: replacing a PDF and re-running regenerates chunks, checksums and lineage |

**Run order:** Cell 1 → upload PDFs → Cell 3 → Cell 4 → Cell 4A → **Cell P (edit config)** → Cell A3 (inspect) → Cell 5A0 → Cell 5A1 (compare) → Cell 5A2 (export) → B3 → B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8A → Cell 9 → Cell 10 (optional evidence).

> No model API key is needed. This notebook creates **prepared text chunks + metadata only** — no embeddings, vector index or LLM calls (those are later topics).

## Part A — Start with the data

### Cell 1 — REQUIRED: Initialise the Colab workspace (run first, and again after any runtime reset)

In [ ]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


**Upload step (manual):** Colab left sidebar → **Files** → `week3_project` → `data` → `raw` → ⋮ → **Upload**.  
Upload the KWSP PDFs you downloaded from the official KWSP website (account restructuring + at least one withdrawal-type document, e.g. Housing and Education).  
Do **not** commit these PDFs to GitHub (README: *source documents are not redistributed*).

### Cell 3 — REQUIRED: Verify uploaded inputs

In [ ]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 5
APENDIX EPF ACCOUNT RESTRUCTURING SET TO ADDRESS MEMBERS LIFE CYCLE NEEDS_25.4.2024 1 (1).pdf bytes: 447312
Account Restructuring FAQ-EN.pdf bytes: 746228
EPF Act 1991_ Legal Framework Overview.pdf bytes: 1355935
KWSP 9FA (AHL)_BORANG PERMOHONAN PENGELUARAN AKAUN FLEKSIBEL_25042024.pdf bytes: 470736
Panduan Mudah Pengeluaran Akaun Fleksibel_ENG.pdf bytes: 2224951
Upload verified. Select your relevant input(s) for Step A3.


### Cell 4 — REQUIRED: Source selection + lineage utilities

In [ ]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]

def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


### Cell 4A — REQUIRED: Initialise the common lineage registry

In [ ]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )

    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )

    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )

    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


### Cell P — EPF-Guide project configuration (**EDIT THIS CELL**)

This cell is the single place where project-specific values live. Every later cell (Part B *and* Part C) reads from it, so both parts use the same preparation configuration.

**Before running, replace the example values with your real ones:**
- `relative_path` → exact PDF filename you uploaded into `data/raw`
- `source_url` → the exact KWSP page/URL you downloaded that PDF from
- `accessed_date` → the date you downloaded it (`YYYY-MM-DD`)
- `document_version` → version/effective date printed on the document or KWSP page; keep `"Not verified"` if none is shown
- `doc_status` → `"current"` only after you have checked on the KWSP website that the document is still the active version. Anything else (e.g. `"superseded"`, `"unverified"`) is **excluded** from preparation — this implements the current-only knowledge source decision (ADR2 / NFR5).

Remove or add dictionaries to match the PDFs you actually have (at least two sources/files are needed for inspection).

In [ ]:
# Cell P — EPF-GUIDE PROJECT CONFIGURATION  (EDIT THE EXAMPLE VALUES)
EPF_SOURCES = [
    {
        "doc_id": "restructuring_faq",
        "relative_path": "Account Restructuring FAQ-EN.pdf",    # Updated with .pdf extension
        "title": "KWSP Account Restructuring FAQ (English)",
        "topic": "Account restructuring policy, objectives, and general rules",
        "source_url": "file:///C:/Users/Kirthiggan/OneDrive/Desktop/KWSP%20PDF/Account%20Restructuring%20FAQ-EN.pdf",            # Official KWSP source
        "accessed_date": "2026-11-12",                       # Current date updated
        "document_version": "Not verified",
        "doc_status": "current",
    },
    {
        "doc_id": "epf_act",
        "relative_path": "EPF Act 1991_ Legal Framework Overview.pdf",
        "title": "Employees Provident Fund Act 1991 - Legal Framework Overview",
        "topic": "Statutory definitions and legal backing",
        "source_url": "file:///C:/Users/Kirthiggan/OneDrive/Desktop/KWSP%20PDF/EPF%20Act%201991_%20Legal%20Framework%20Overview.pdf",
        "accessed_date": "2026-11-12",
        "document_version": "Not verified",
        "doc_status": "current",
    },
    {
        "doc_id": "lifecycle_appendix",
        "relative_path": "APENDIX EPF ACCOUNT RESTRUCTURING SET TO ADDRESS MEMBERS LIFE CYCLE NEEDS_25.4.2024 1 (1).pdf",
        "title": "EPF Account Restructuring Life Cycle Needs Appendix",
        "topic": "Policy rationale addressing members' life cycle needs",
        "source_url": "file:///C:/Users/Kirthiggan/OneDrive/Desktop/KWSP%20PDF/APENDIX%20EPF%20ACCOUNT%20RESTRUCTURING%20SET%20TO%20ADDRESS%20MEMBERS%20LIFE%20CYCLE%20NEEDS_25.4.2024%201%20(1).pdf",
        "accessed_date": "2026-11-12",
        "document_version": "2024-04-25",
        "doc_status": "current",
    },
    {
        "doc_id": "fleksibel_guide",
        "relative_path": "Panduan Mudah Pengeluaran Akaun Fleksibel_ENG.pdf",
        "title": "Akaun Fleksibel Withdrawal Quick Guide (English)",
        "topic": "Step-by-step procedure for Akaun Fleksibel withdrawals",
        "source_url": "file:///C:/Users/Kirthiggan/OneDrive/Desktop/KWSP%20PDF/Panduan%20Mudah%20Pengeluaran%20Akaun%20Fleksibel_ENG.pdf",
        "accessed_date": "2026-11-12",
        "document_version": "Not verified",
        "doc_status": "current",
    },
    {
        "doc_id": "form_9fa",
        "relative_path": "KWSP 9FA (AHL)_BORANG PERMOHONAN PENGELUARAN AKAUN FLEKSIBEL_25042024.pdf",
        "title": "KWSP Form 9FA - Akaun Fleksibel Withdrawal Application Form",
        "topic": "Official application form rules and requirements",
        "source_url": "file:///C:/Users/Kirthiggan/OneDrive/Desktop/KWSP%20PDF/KWSP%209FA%20(AHL)_BORANG%20PERMOHONAN%20PENGELUARAN%20AKAUN%20FLEKSIBEL_25042024.pdf",
        "accessed_date": "2026-11-12",
        "document_version": "2024-04-25",
        "doc_status": "current",
    },
]

# ---- Preparation configuration (used identically in Part B and Part C) ----
CHUNK_SIZE = 800                # characters per chunk (same size for both compared methods)
CHOSEN_METHOD = "page-aware"    # "page-aware" or "fixed"  (decide after Cell 5A1 evidence)
REMOVE_REPEATED_LINES = True    # Set True to clean up form boilerplate and headers across documents
MIN_PAGE_CHARS = 20             # a page with fewer extracted characters is flagged as empty
MIN_CHUNK_CHARS = 40            # shorter chunks are marked status="needs_review"

# ---- Safety checks on the configuration ----
INCLUDE_STATUSES = {"current"}
ACTIVE_SOURCES = [d for d in EPF_SOURCES if d["doc_status"] in INCLUDE_STATUSES]
EXCLUDED_SOURCES = [d for d in EPF_SOURCES if d["doc_status"] not in INCLUDE_STATUSES]

for d in EXCLUDED_SOURCES:
    print(f"EXCLUDED (status={d['doc_status']!r}): {d['relative_path']}")

assert ACTIVE_SOURCES, "No source has doc_status='current'. Verify documents on the KWSP site first."
for d in ACTIVE_SOURCES:
    assert d["accessed_date"] != "YYYY-MM-DD", (
        f"Fill in the real accessed_date for {d['relative_path']} before running."
    )
    assert (RAW_DIR / d["relative_path"]).is_file(), (
        f"Not found under data/raw: {d['relative_path']}. "
        "Fix relative_path to match the uploaded filename (see Cell 3 listing)."
    )

EPF_META_BY_NAME = {d["relative_path"]: d for d in ACTIVE_SOURCES}
print("\nActive sources:", [d["relative_path"] for d in ACTIVE_SOURCES])
print("Chunk size:", CHUNK_SIZE, "| Method:", CHOSEN_METHOD,
      "| Remove repeated lines:", REMOVE_REPEATED_LINES)


Active sources: ['Account Restructuring FAQ-EN.pdf', 'EPF Act 1991_ Legal Framework Overview.pdf', 'APENDIX EPF ACCOUNT RESTRUCTURING SET TO ADDRESS MEMBERS LIFE CYCLE NEEDS_25.4.2024 1 (1).pdf', 'Panduan Mudah Pengeluaran Akaun Fleksibel_ENG.pdf', 'KWSP 9FA (AHL)_BORANG PERMOHONAN PENGELUARAN AKAUN FLEKSIBEL_25042024.pdf']
Chunk size: 800 | Method: page-aware | Remove repeated lines: True


### Cell A3 — REQUIRED: Inspect what was actually loaded

Evidence for the Data Inventory (`docs/week3_data_inventory.md`, Step A4). A file existing is **not** evidence of successful ingestion — this cell shows pages loaded, characters per page, empty pages, a text sample, and possible repeated header/footer lines for **each** source (at least two).

In [ ]:
# Cell A3 — Inspect selected PDF sources (REQUIRED)
!pip -q install pypdf
import json, re
from collections import Counter
import pypdf
from pypdf import PdfReader

print("pypdf version:", pypdf.__version__)

def extract_pages(path):
    """Return a list of (page_number, raw_text) with 1-based page numbers."""
    reader = PdfReader(str(path))
    return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]

def repeated_line_candidates(pages, min_share=0.6, min_pages=3, max_len=80):
    """Short lines that repeat on most pages (likely headers/footers). Candidates only."""
    if len(pages) < min_pages:
        return []
    counter = Counter()
    for _, text in pages:
        seen = {ln.strip() for ln in text.splitlines() if ln.strip() and len(ln.strip()) <= max_len}
        counter.update(seen)
    threshold = max(min_pages, int(len(pages) * min_share))
    return sorted([ln for ln, n in counter.items() if n >= threshold])

inspection_summary = []
for d in ACTIVE_SOURCES:
    path = select_source(relative_path=d["relative_path"])
    pages = extract_pages(path)
    chars = [len(t.strip()) for _, t in pages]
    empty_pages = [n for (n, t) in pages if len(t.strip()) < MIN_PAGE_CHARS]
    first_text = next((t for _, t in pages if t.strip()), "")
    candidates = repeated_line_candidates(pages)

    print("=" * 70)
    print("Source:", d["relative_path"], "|", d["title"])
    print("Bytes:", path.stat().st_size, "| SHA-256:", source_hash(path)[:16], "...")
    print("Pages loaded:", len(pages), "| Total characters:", sum(chars))
    print("Characters per page (first 10):", chars[:10])
    print("Empty / near-empty pages (<", MIN_PAGE_CHARS, "chars):", empty_pages)
    print("Repeated-line candidates (header/footer?):", candidates[:8])
    print("Text sample:", repr(first_text.strip()[:300]))

    assert sum(chars) > 0, f"No extractable text in {path.name}. Scanned PDF? OCR is outside this lab."
    inspection_summary.append({
        "relative_path": d["relative_path"],
        "pages": len(pages),
        "total_chars": sum(chars),
        "empty_pages": empty_pages,
        "repeated_line_candidates": candidates[:20],
    })

summary_path = BASE_DIR / "data" / "week3_inspection.json"
summary_path.write_text(json.dumps(inspection_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nInspection summary saved:", summary_path)
print("Sources inspected:", len(inspection_summary))

pypdf version: 7.0.1
Source: Account Restructuring FAQ-EN.pdf | KWSP Account Restructuring FAQ (English)
Bytes: 746228 | SHA-256: 15305d58af7b0d9b ...
Pages loaded: 11 | Total characters: 14165
Characters per page (first 10): [2008, 1565, 965, 995, 516, 69, 1582, 1919, 2352, 2124]
Empty / near-empty pages (< 20 chars): []
Repeated-line candidates (header/footer?): ['Final Version V1.1 (29042024)', 'RESTRICTED']
Text sample: 'Page 1 of 11 \nFinal Version V1.1 (29042024) \n \nRESTRICTED \nRESTRICTED \nFREQUENTLY ASKED QUESTION (FAQs) ON EPF ACCOUNT RESTRUCTURING \n \nNO. BASIC CONCEPT \n1. What is EPF account restructuring? \n \nThe EPF account restructuring aims to enhance members’ income security after retirement \nwhile addressin'
Source: EPF Act 1991_ Legal Framework Overview.pdf | Employees Provident Fund Act 1991 - Legal Framework Overview
Bytes: 1355935 | SHA-256: 81671aae6db7bde2 ...
Pages loaded: 49 | Total characters: 157184
Characters per page (first 10): [2590, 4290, 3565, 28

> **Record in the inventory (Step A4):** for each source write pages loaded, total characters, empty pages, and one short text sample. If a page extracts as empty, note it as an ingestion limitation (scanned/image-only page → OCR is outside this lab).
> If a **table** (e.g. eligibility conditions) looks flattened or out of order in the sample, note that too — it affects FR2 later.

## Part B — Prepare the data (Pathway A: PDF → page-aware text chunks)

### Cell 5A0 — Shared document-preparation helpers (REQUIRED for Pathway A)

Both Part B (this section) and Part C (Cell 8A) call these same functions, so the preparation configuration cannot silently differ between the two parts.

**Cleaning rule (justified, minimal):** collapse repeated spaces/tabs and 3+ blank lines. Repeated header/footer lines are removed **only** if `REMOVE_REPEATED_LINES=True`, i.e. only after you have inspected the candidates in Cell A3.

In [ ]:
# Cell 5A0 — Shared helpers for document preparation
import re

def clean_page_text(text, drop_lines=None):
    drop = set(drop_lines or [])
    lines = [ln for ln in text.splitlines() if ln.strip() not in drop]
    text = "\n".join(lines)
    text = re.sub(r"[ \t]+", " ", text)          # collapse spaces/tabs
    text = re.sub(r"\n[ \t]*\n(?:[ \t]*\n)+", "\n\n", text)   # 3+ blank lines -> 1 blank line
    return text.strip()

def split_long(segment, size):
    """Split an over-long segment at line, then word boundaries (never mid-word if avoidable)."""
    parts, current = [], ""
    for line in segment.splitlines():
        line = line.strip()
        if not line:
            continue
        while len(line) > size:                    # single very long line: cut at last space
            cut = line.rfind(" ", 0, size)
            cut = cut if cut > 0 else size
            if current:
                parts.append(current)
                current = ""
            parts.append(line[:cut].strip())
            line = line[cut:].strip()
        if not line:
            continue
        if len(current) + len(line) + (1 if current else 0) <= size:
            current += ("\n" if current else "") + line
        else:
            if current:
                parts.append(current)
            current = line
    if current:
        parts.append(current)
    return parts

def fixed_chunks(s, size):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size):
    """Paragraph-aware packing within ONE page; falls back to line/word splitting."""
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts, current = [], ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(split_long(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

def load_document_pages(path, remove_repeated):
    """Extract pages and apply the justified cleaning. Returns (pages, skipped_empty_pages, dropped_lines)."""
    raw_pages = extract_pages(path)
    drop = repeated_line_candidates(raw_pages) if remove_repeated else []
    pages, skipped = [], []
    for n, text in raw_pages:
        cleaned = clean_page_text(text, drop_lines=drop)
        if len(cleaned) < MIN_PAGE_CHARS:
            skipped.append(n)
        else:
            pages.append((n, cleaned))
    return pages, skipped, drop

def chunk_document(pages, method, size):
    """Return list of (page_or_None, chunk_text)."""
    if method == "page-aware":
        out = []
        for n, text in pages:
            out.extend((n, c) for c in paragraph_chunks(text, size))
        return out
    elif method == "fixed":
        whole = "\n".join(t for _, t in pages)
        return [(None, c) for c in fixed_chunks(whole, size)]   # page boundaries are lost
    raise ValueError("method must be 'page-aware' or 'fixed'")

def build_records(meta_list, method, size, remove_repeated):
    """Build traceable chunk records for all active documents (deterministic: no timestamps)."""
    records, notes = [], []
    for meta in meta_list:
        path = RAW_DIR / meta["relative_path"]
        sha = source_hash(path)
        pages, skipped, dropped = load_document_pages(path, remove_repeated)
        chunks = chunk_document(pages, method, size)
        notes.append({"doc_id": meta["doc_id"], "pages_used": len(pages),
                      "empty_pages_skipped": skipped, "repeated_lines_removed": dropped,
                      "chunks": len(chunks)})
        per_page_counter = {}
        for page, text in chunks:
            key = page if page is not None else 0
            per_page_counter[key] = per_page_counter.get(key, 0) + 1
            suffix = f"p{page:03d}" if page is not None else "pXXX"
            records.append({
                "chunk_id": f"{meta['doc_id']}-{suffix}-{per_page_counter[key]:03d}"
                            if page is not None else f"{meta['doc_id']}-{len(records)+1:05d}",
                "text": text,
                "source": meta["relative_path"],
                "source_title": meta["title"],
                "source_url": meta["source_url"],
                "source_sha256": sha,
                "accessed_date": meta["accessed_date"],
                "document_version": meta["document_version"],
                "doc_status": meta["doc_status"],
                "page": page,
                "processing": f"{method}-char-{size}; clean=collapse-whitespace"
                              f"{'; repeated-lines-removed' if remove_repeated else ''}",
                "status": "ready" if len(text.strip()) >= MIN_CHUNK_CHARS else "needs_review",
            })
    return records, notes

print("Cell 5A0 ready: helpers defined.")

Cell 5A0 ready: helpers defined.


### Cell 5A1 — Compare two preparation approaches on the same input (Step B3 evidence)

| | Approach A | Approach B |
|---|---|---|
| Method | **fixed-size** characters over the whole concatenated document | **page-aware paragraph/line-aware** chunks (one page at a time) |
| Size | `CHUNK_SIZE` | `CHUNK_SIZE` (same, to isolate the strategy) |

The metrics below are simple, observable **proxies**, not a full quality assessment (that is Topic 4): chunk count, length statistics, share of chunks that **keep a page number** (needed for FR3 / RAI1), and share of chunks that **start or end mid-sentence**.

In [ ]:
# Cell 5A1 — Compare fixed-size vs page-aware chunking (same sources, same size)
import pandas as pd

def midsentence_stats(records):
    starts = sum(1 for r in records if r["text"][:1].islower())
    ends = sum(1 for r in records if r["text"].rstrip()[-1:] not in ".:;!?)\u201d\"")
    n = max(len(records), 1)
    return round(100 * starts / n, 1), round(100 * ends / n, 1)

comparison_rows = []
approach_records = {}
for method in ["fixed", "page-aware"]:
    recs, notes = build_records(ACTIVE_SOURCES, method, CHUNK_SIZE, REMOVE_REPEATED_LINES)
    approach_records[method] = recs
    lengths = [len(r["text"]) for r in recs]
    with_page = sum(1 for r in recs if r["page"] is not None)
    s, e = midsentence_stats(recs)
    comparison_rows.append({
        "approach": method,
        "chunks": len(recs),
        "min_len": min(lengths), "mean_len": round(sum(lengths) / len(lengths), 1), "max_len": max(lengths),
        "chunks_with_page_%": round(100 * with_page / len(recs), 1),
        "start_midsentence_%": s,
        "end_nonterminal_%": e,
        "needs_review": sum(1 for r in recs if r["status"] == "needs_review"),
    })

comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False))

print("\n--- Fixed sample (first chunk) ---")
print(repr(approach_records["fixed"][0]["text"][:200]))
print("\n--- Page-aware sample (first chunk) ---")
print({k: approach_records["page-aware"][0][k] for k in ["chunk_id", "source", "page"]})
print(repr(approach_records["page-aware"][0]["text"][:200]))
print("\nCopy this table into docs/week3_pipeline_decisions.md (Part B comparison).")

  approach  chunks  min_len  mean_len  max_len  chunks_with_page_%  start_midsentence_%  end_nonterminal_%  needs_review
     fixed     243       27     792.7      800                 0.0                 72.4               96.3             1
page-aware     319       20     601.4      800               100.0                 21.0               71.2             6

--- Fixed sample (first chunk) ---
'Page 1 of 11 \n \nFREQUENTLY ASKED QUESTION (FAQs) ON EPF ACCOUNT RESTRUCTURING \n \nNO. BASIC CONCEPT \n1. What is EPF account restructuring? \n \nThe EPF account restructuring aims to enhance members’ inco'

--- Page-aware sample (first chunk) ---
{'chunk_id': 'restructuring_faq-p001-001', 'source': 'Account Restructuring FAQ-EN.pdf', 'page': 1}
'Page 1 of 11\n\nFREQUENTLY ASKED QUESTION (FAQs) ON EPF ACCOUNT RESTRUCTURING\n\nNO. BASIC CONCEPT \n1. What is EPF account restructuring?\n\nThe EPF account restructuring aims to enhance members’ income sec'

Copy this table into docs/week3_pipelin

> **Decide:** read the table and the two samples. For EPF-Guide, FR3/RAI1 need a **page number on every chunk**, so a method with `chunks_with_page_% = 0` cannot support page-level citation. If your evidence supports it, keep `CHOSEN_METHOD = "page-aware"` in Cell P; otherwise change it and justify the change in `docs/week3_pipeline_decisions.md`.

### Cell 5A2 — REQUIRED for Pathway A: export traceable chunk records (`prepared_text.jsonl`)

In [ ]:
# Cell 5A2 — Export prepared records using the CHOSEN method
import json

records, build_notes = build_records(ACTIVE_SOURCES, CHOSEN_METHOD, CHUNK_SIZE, REMOVE_REPEATED_LINES)
approach = f"{CHOSEN_METHOD}-char-{CHUNK_SIZE}"

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"

# Project-specific checks (FR3 / RAI1 / NFR5)
for d in ACTIVE_SOURCES:
    assert any(r["source"] == d["relative_path"] for r in records), f"No chunks from {d['relative_path']}"
if CHOSEN_METHOD == "page-aware":
    assert all(isinstance(r["page"], int) and r["page"] >= 1 for r in records), "A chunk lacks a page number"
assert all(r["doc_status"] == "current" for r in records), "Non-current document leaked into output"

print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
for note in build_notes:
    print("  ", note)
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", {k: v for k, v in records[0].items() if k != "text"})
print("First record text:", repr(records[0]["text"][:200]))

PATHWAY A2 completed successfully.
Selected approach: page-aware-char-800
Total chunks exported: 319
   {'doc_id': 'restructuring_faq', 'pages_used': 9, 'empty_pages_skipped': [6, 11], 'repeated_lines_removed': ['Final Version V1.1 (29042024)', 'RESTRICTED'], 'chunks': 22}
   {'doc_id': 'epf_act', 'pages_used': 49, 'empty_pages_skipped': [], 'repeated_lines_removed': ['11/12/26, 1:48 PM EPF Act 1991: Legal Framework Overview'], 'chunks': 260}
   {'doc_id': 'lifecycle_appendix', 'pages_used': 2, 'empty_pages_skipped': [], 'repeated_lines_removed': [], 'chunks': 6}
   {'doc_id': 'fleksibel_guide', 'pages_used': 2, 'empty_pages_skipped': [], 'repeated_lines_removed': [], 'chunks': 3}
   {'doc_id': 'form_9fa', 'pages_used': 4, 'empty_pages_skipped': [], 'repeated_lines_removed': ['9FA (AHL)', 'For any inquiries:', 'KWSP', 'Laman Web Rasmi KWSP / EPF Official Website : www.kwsp.gov.my', 'PENGELUARAN AKAUN FLEKSIBEL', 'Pusat Pengurusan Perhubungan KWSP / EPF Contact Management Centre : 03-89

### B3 — REQUIRED: basic output contract (run after the pathway)

In [ ]:
# B3 BASIC OUTPUT CONTRACT — REQUIRED
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

# Manual content check: look at three records from different positions
import json
with prepared_path.open(encoding="utf-8") as fh:
    rows = [json.loads(l) for l in fh if l.strip()]
for idx in sorted({0, len(rows) // 2, len(rows) - 1}):
    r = rows[idx]
    print(f"\n[{idx}] {r['chunk_id']} | {r['source']} | page {r['page']} | status {r['status']}")
    print(r["text"][:300])

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 390012
Validation: file exists and is non-empty; verify content manually too.

[0] restructuring_faq-p001-001 | Account Restructuring FAQ-EN.pdf | page 1 | status ready
Page 1 of 11

FREQUENTLY ASKED QUESTION (FAQs) ON EPF ACCOUNT RESTRUCTURING

NO. BASIC CONCEPT 
1. What is EPF account restructuring?

The EPF account restructuring aims to enhance members’ income security after retirement 
while addressing their life cycle needs.

The account will be restructured f

[159] epf_act-p025-006 | EPF Act 1991_ Legal Framework Overview.pdf | page 25 | status ready
and if the Board approves such transfer, the employee shall be credited with the amount so transferred to the Fund.
(15) The provisions of section 51 shall apply in respect of an approved fund in like manner as it applies in respect of the Fund.
(16) For the purpose of this section the expression "a

[318] form_9fa-p004-007 | KWSP 9FA (AHL)_BORANG PERMOHONAN PEN

### B4.1 — Register lineage for every selected source (Pathway A)

In [ ]:
# Register Pathway A — PDF/TXT (one registration per source document)
for d in ACTIVE_SOURCES:
    register_lineage(
        source=RAW_DIR / d["relative_path"],
        output=prepared_path,
        processing=(
            f"{approach}; clean=collapse-whitespace; "
            f"repeated_lines_removed={REMOVE_REPEATED_LINES}; page_metadata_preserved="
            f"{CHOSEN_METHOD == 'page-aware'}"
        )
    )
print("Registered entries:", len(lineage_entries))

Registered: Account Restructuring FAQ-EN.pdf -> prepared_text.jsonl
Registered: EPF Act 1991_ Legal Framework Overview.pdf -> prepared_text.jsonl
Registered: APENDIX EPF ACCOUNT RESTRUCTURING SET TO ADDRESS MEMBERS LIFE CYCLE NEEDS_25.4.2024 1 (1).pdf -> prepared_text.jsonl
Registered: Panduan Mudah Pengeluaran Akaun Fleksibel_ENG.pdf -> prepared_text.jsonl
Registered: KWSP 9FA (AHL)_BORANG PERMOHONAN PENGELUARAN AKAUN FLEKSIBEL_25042024.pdf -> prepared_text.jsonl
Registered entries: 5


### Cell 6A — REQUIRED: generate the common lineage manifest

In [ ]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)

lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    rel_name = str(source.relative_to(RAW_DIR.resolve()))
    meta = EPF_META_BY_NAME.get(rel_name, {})      # EPF-Guide: carry project metadata into lineage

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": rel_name,
        "source_title": meta.get("title", "Not recorded"),
        "source_url": meta.get("source_url", "Not recorded"),
        "accessed_date": meta.get("accessed_date", "Not recorded"),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": meta.get("document_version", "Not verified"),
        "doc_status": meta.get("doc_status", "Not recorded"),
        "access_rights": (
            "Public KWSP document; source PDF not redistributed in repo; "
            "reuse terms not verified"
        ),
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 5
Saved to: /content/week3_project/data/week3_manifest.json


### Cell 6B — REQUIRED: verify the saved manifest

In [ ]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 5
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — Build and run the V1 data pipeline

### Cell 7 — REQUIRED: Common V1 pipeline utilities

Identical to the lab template **except one small documented adaptation**: for `kind == "documents"` the pipeline accepts a *list* of explicitly selected PDF files (EPF-Guide has several KWSP documents, not one). Images and all other kinds behave exactly as in the template.

In [ ]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}

# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )
    return source

# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # INGEST: Identify relevant source files
    if kind == "images":
        source = Path(source)
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    elif kind == "documents" and isinstance(source, (list, tuple)):
        # EPF-Guide adaptation: several explicitly selected KWSP PDFs
        inputs = [Path(p) for p in source]
    else:
        inputs = [Path(source)]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )
    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )
        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} │ "
        f"Inputs: {len(inputs)} │ "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)
    return record

# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()
    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


### Cell 8A — CONDITIONAL REQUIRED: PDF documents (EPF-Guide)

Same sources and same configuration as Part B (read from Cell P, using the helpers from Cell 5A0). Output: `data/pipeline_v1/documents/prepared_text.jsonl` + `pipeline_manifest.json`.

In [ ]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects (EPF-Guide)
# Requires Cell 7, Cell P and Cell 5A0 (same preparation configuration as Part B).
import json

# Step 1: Select the sources and the Part B configuration
SOURCE_A = [resolve_source(d["relative_path"]) for d in ACTIVE_SOURCES]
CONFIG_A = {
    "method": CHOSEN_METHOD,
    "chunk_size": CHUNK_SIZE,
    "remove_repeated_lines": REMOVE_REPEATED_LINES,
    "min_page_chars": MIN_PAGE_CHARS,
    "documents": ACTIVE_SOURCES,
}

# Step 2: Inspect document content (stage contract: readable text + expected properties)
def inspect_documents(inputs, cfg):
    details = []
    for src in inputs:
        pages = extract_pages(src)
        total = sum(len(t.strip()) for _, t in pages)
        empty = [n for n, t in pages if len(t.strip()) < cfg["min_page_chars"]]
        if total == 0:
            raise ValueError(
                f"No extractable text in {src.name}. Scanned PDFs may require OCR."
            )
        details.append({"source": src.name, "pages": len(pages),
                        "characters": total, "empty_pages": empty})
    return {"documents": details}

# Step 3: Prepare and store records
def prepare_documents(inputs, out, cfg):
    meta_by_path = {str(RAW_DIR / d["relative_path"]): d for d in cfg["documents"]}
    meta_list = [meta_by_path[str(p)] for p in inputs]
    records, notes = build_records(
        meta_list, cfg["method"], int(cfg["chunk_size"]), cfg["remove_repeated_lines"]
    )
    if not records:
        raise ValueError("No text chunks produced")

    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    mapping = [{"source": str(p), "output": str(target)} for p in inputs]
    method = (f"{cfg['method']}-char-{cfg['chunk_size']}; clean=collapse-whitespace; "
              f"repeated_lines_removed={cfg['remove_repeated_lines']}")
    return [target], mapping, method

# Step 4: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")
print("Inspection:", run_A["inspection"])

SUCCESS: documents │ Inputs: 5 │ Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.
Inspection: {'documents': [{'source': 'Account Restructuring FAQ-EN.pdf', 'pages': 11, 'characters': 14165, 'empty_pages': []}, {'source': 'EPF Act 1991_ Legal Framework Overview.pdf', 'pages': 49, 'characters': 157184, 'empty_pages': []}, {'source': 'APENDIX EPF ACCOUNT RESTRUCTURING SET TO ADDRESS MEMBERS LIFE CYCLE NEEDS_25.4.2024 1 (1).pdf', 'pages': 2, 'characters': 2954, 'empty_pages': []}, {'source': 'Panduan Mudah Pengeluaran Akaun Fleksibel_ENG.pdf', 'pages': 2, 'characters': 1229, 'empty_pages': []}, {'source': 'KWSP 9FA (AHL)_BORANG PERMOHONAN PENGELUARAN AKAUN FLEKSIBEL_25042024.pdf', 'pages': 4, 'characters': 21540, 'empty_pages': []}]}


### Cell 9 — REQUIRED: repeatability and provenance checks

In [ ]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)

print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents │ Inputs: 5 │ Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


### Cell 10 — OPTIONAL: Part B vs Part C consistency + shareable evidence

1. Confirms that the Part B export and the Part C export contain the **same chunks** (the lab asks you to explain any differences).
2. Writes `data/evidence/prepared_text_preview.json`: record counts per source plus **truncated** text previews only. Use this as non-sensitive execution evidence if you decide *not* to commit the full KWSP text (see the rights note in `docs/week3_pipeline_decisions.md`).

In [ ]:
# Cell 10 — OPTIONAL consistency check + shareable evidence preview
import json

part_b = [json.loads(l) for l in (PROCESSED_DIR / "prepared_text.jsonl").open(encoding="utf-8") if l.strip()]
part_c = [json.loads(l) for l in (PIPE_DIR / "documents" / "prepared_text.jsonl").open(encoding="utf-8") if l.strip()]

print("Part B chunks:", len(part_b), "| Part C chunks:", len(part_c))
print("Identical records in both parts:", part_b == part_c)

per_source = {}
for r in part_c:
    per_source.setdefault(r["source"], 0)
    per_source[r["source"]] += 1

preview = {
    "pipeline_version": "week3-v1",
    "method": CHOSEN_METHOD,
    "chunk_size": CHUNK_SIZE,
    "chunks_per_source": per_source,
    "needs_review": sum(1 for r in part_c if r["status"] == "needs_review"),
    "sample_records": [
        {**{k: v for k, v in r.items() if k != "text"}, "text_preview": r["text"][:120] + "..."}
        for r in part_c[:3]
    ],
}
evidence_dir = BASE_DIR / "data" / "evidence"
evidence_dir.mkdir(parents=True, exist_ok=True)
(evidence_dir / "prepared_text_preview.json").write_text(
    json.dumps(preview, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", evidence_dir / "prepared_text_preview.json")
print(json.dumps(preview["chunks_per_source"], indent=2))

Part B chunks: 319 | Part C chunks: 319
Identical records in both parts: True
Saved: /content/week3_project/data/evidence/prepared_text_preview.json
{
  "Account Restructuring FAQ-EN.pdf": 22,
  "EPF Act 1991_ Legal Framework Overview.pdf": 260,
  "APENDIX EPF ACCOUNT RESTRUCTURING SET TO ADDRESS MEMBERS LIFE CYCLE NEEDS_25.4.2024 1 (1).pdf": 6,
  "Panduan Mudah Pengeluaran Akaun Fleksibel_ENG.pdf": 3,
  "KWSP 9FA (AHL)_BORANG PERMOHONAN PENGELUARAN AKAUN FLEKSIBEL_25042024.pdf": 28
}


## Downloads for GitHub (Step C4a)
Download from the Colab Files panel: `week3_pipeline.ipynb` (File → Download → .ipynb), `data/week3_manifest.json`, `data/week3_inspection.json`, `data/processed/prepared_text.jsonl`, `data/pipeline_v1/documents/` (outputs + `pipeline_manifest.json`) and `data/evidence/prepared_text_preview.json`.  
Files under `/content` are temporary — download before the runtime resets.